In [ ]:
# ==============================================================================
# PROYECTO: MOTOR DE RIESGO CREDITICIO Y OPTIMIZACIÓN DE RENTABILIDAD
# AUTOR: [Tu Nombre] - Data Scientist
# ENFOQUE: In-Database Processing (DuckDB), Monotonic XGBoost, P&L Optimization
# ==============================================================================

import os
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import xgboost as xgb
import shap

from scipy.stats.mstats import winsorize
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from sklearn.impute import SimpleImputer
import warnings

# 0. CONFIGURACIÓN INICIAL
# ------------------------------------------------------------------------------
plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')

# RUTAS (Ajustar en Google Colab)
from google.colab import drive
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/Colab Notebooks/Predicción de crédito/'
application = base_path + 'application_train.csv'
bureau = base_path + 'bureau.csv'
payments = base_path + 'installments_payments.csv'
previous = base_path + 'previous_application.csv'

# 1. DATA ENGINEERING (IN-DATABASE CON DUCKDB)
# ------------------------------------------------------------------------------
print("⏳ 1. Procesando variables de comportamiento en DuckDB...")
con = duckdb.connect(database=':memory:')

query_etl = f"""
CREATE OR REPLACE VIEW bureau_window AS
WITH ranked_bureau AS (
    SELECT *, ROW_NUMBER() OVER(PARTITION BY SK_ID_CURR ORDER BY DAYS_CREDIT DESC) as recency
    FROM read_csv_auto('{bureau}')
)
SELECT SK_ID_CURR, SUM(AMT_CREDIT_SUM_DEBT) AS deuda_externa_total,
       MAX(AMT_CREDIT_SUM_OVERDUE) AS mora_max_externa,
       MAX(CASE WHEN recency = 1 THEN CREDIT_ACTIVE ELSE 'Desconocido' END) AS estado_ultimo_credito_ext
FROM ranked_bureau GROUP BY SK_ID_CURR;

CREATE OR REPLACE VIEW install_window AS
WITH ranked_payments AS (
    SELECT SK_ID_CURR, (AMT_PAYMENT / NULLIF(AMT_INSTALMENT, 0)) AS ratio_pago,
           (DAYS_ENTRY_PAYMENT - DAYS_INSTALMENT) AS delay,
           ROW_NUMBER() OVER(PARTITION BY SK_ID_CURR ORDER BY DAYS_INSTALMENT DESC) as recency
    FROM read_csv_auto('{payments}')
)
SELECT SK_ID_CURR, AVG(delay) AS promedio_dias_atraso_hist,
       AVG(delay) FILTER (WHERE recency <= 3) AS atraso_reciente_3m,
       AVG(ratio_pago) FILTER (WHERE recency <= 3) AS cumplimiento_reciente_3m
FROM ranked_payments GROUP BY SK_ID_CURR;

CREATE OR REPLACE VIEW prev_agg AS
SELECT SK_ID_CURR, AVG(CASE WHEN NAME_CONTRACT_STATUS = 'Refused' THEN 1 ELSE 0 END) AS tasa_rechazo_interno
FROM read_csv_auto('{previous}') GROUP BY SK_ID_CURR;

SELECT
    app.SK_ID_CURR, app.TARGET, app.CODE_GENDER, app.FLAG_OWN_CAR, app.FLAG_OWN_REALTY,
    app.AMT_INCOME_TOTAL AS ingreso_total, app.AMT_CREDIT AS monto_credito, app.AMT_ANNUITY AS monto_anualidad,
    app.DAYS_BIRTH / -365.0 AS edad_años, app.DAYS_EMPLOYED AS dias_empleado_raw,
    app.EXT_SOURCE_1, app.EXT_SOURCE_2, app.EXT_SOURCE_3,
    COALESCE(b.deuda_externa_total, 0) AS deuda_externa_total, COALESCE(b.mora_max_externa, 0) AS mora_max_externa,
    COALESCE(b.estado_ultimo_credito_ext, 'Sin_Historial') AS estado_ultimo_credito_ext,
    COALESCE(i.promedio_dias_atraso_hist, 0) AS promedio_dias_atraso_hist, COALESCE(i.atraso_reciente_3m, 0) AS atraso_reciente_3m,
    COALESCE(p.tasa_rechazo_interno, 0) AS tasa_rechazo_interno
FROM read_csv_auto('{application}') app
LEFT JOIN bureau_window b ON app.SK_ID_CURR = b.SK_ID_CURR
LEFT JOIN install_window i ON app.SK_ID_CURR = i.SK_ID_CURR
LEFT JOIN prev_agg p ON app.SK_ID_CURR = p.SK_ID_CURR
"""
df_master = con.execute(query_etl).df()

# 2. FEATURE ENGINEERING Y SANEAMIENTO
# ------------------------------------------------------------------------------
print("⏳ 2. Saneamiento Estadístico y Ratios Financieros...")
df_master['FLAG_PENSIONADO'] = np.where(df_master['dias_empleado_raw'] == 365243, 1, 0)
df_master['años_empleado'] = df_master['dias_empleado_raw'].replace(365243, 0).abs() / 365.0
df_master.drop(columns=['dias_empleado_raw'], inplace=True)

df_master['RATIO_APALANCAMIENTO'] = (df_master['deuda_externa_total'] + df_master['monto_credito']) / (df_master['ingreso_total'] + 1)
df_master['RATIO_CARGA_CUOTA'] = df_master['monto_anualidad'] / ((df_master['ingreso_total'] / 12) + 1)
df_master['DELTA_ATRASO'] = df_master['atraso_reciente_3m'] - df_master['promedio_dias_atraso_hist']

for col in['ingreso_total', 'deuda_externa_total', 'DELTA_ATRASO']:
    df_master[col] = winsorize(df_master[col], limits=[0, 0.01])

df_master = pd.get_dummies(df_master, drop_first=True)

# 3. SPLIT Y ENTRENAMIENTO DEL MODELO CAMPEÓN (XGBOOST)
# ------------------------------------------------------------------------------
print("⏳ 3. Entrenando Modelo con Restricciones Monotónicas...")
X = df_master.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df_master['TARGET']
montos_credito = df_master['monto_credito']

X_train, X_test, y_train, y_test, _, montos_test = train_test_split(
    X, y, montos_credito, test_size=0.20, random_state=42, stratify=y
)

imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

constraints = {
    'ingreso_total': -1, 'EXT_SOURCE_1': -1, 'EXT_SOURCE_2': -1, 'EXT_SOURCE_3': -1,
    'deuda_externa_total': 1, 'RATIO_APALANCAMIENTO': 1, 'RATIO_CARGA_CUOTA': 1,
    'tasa_rechazo_interno': 1, 'DELTA_ATRASO': 1
}
monotone_tuple = tuple([constraints.get(col, 0) for col in X_train_imp.columns])

model_xgb = xgb.XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=5,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    monotone_constraints=monotone_tuple, n_jobs=-1, random_state=42
)
model_xgb.fit(X_train_imp, y_train)
y_probs = model_xgb.predict_proba(X_test_imp)[:, 1]

# 4. EVALUACIÓN ESTADÍSTICA (CURVA KS)
# ------------------------------------------------------------------------------
fpr, tpr, thresholds = roc_curve(y_test, y_probs)
auc = roc_auc_score(y_test, y_probs)
gini = (2 * auc) - 1

# Cálculo del estadístico KS
ks_distances = tpr - fpr
ks_idx = np.argmax(ks_distances)
ks_stat = ks_distances[ks_idx]
ks_threshold = thresholds[ks_idx]

# 5. EVALUACIÓN FINANCIERA (P&L OPTIMIZATION)
# ------------------------------------------------------------------------------
TASA_INTERES = 0.15  # Ganancia del 15% sobre préstamos sanos
LGD = 0.60           # Pérdida del 60% sobre el capital en caso de Default

def calcular_profit_detallado(y_true, y_probs, montos, umbral):
    aprobados = (y_probs < umbral)
    rechazados = (y_probs >= umbral)

    ganancia = montos[aprobados & (y_true == 0)].sum() * TASA_INTERES
    perdida = montos[aprobados & (y_true == 1)].sum() * LGD
    costo_op = montos[rechazados & (y_true == 0)].sum() * TASA_INTERES
    capital_salvado = montos[rechazados & (y_true == 1)].sum() * LGD

    return ganancia - perdida, ganancia, perdida, costo_op, capital_salvado

umbrales_busqueda = np.linspace(0.01, 0.50, 50)
resultados_fin =[calcular_profit_detallado(y_test, y_probs, montos_test.values, t) for t in umbrales_busqueda]

beneficios = [res[0] for res in resultados_fin]
idx_optimo = np.argmax(beneficios)
umbral_optimo = umbrales_busqueda[idx_optimo]
net_profit, ganancia_int, perdida_def, costo_op, cap_salvado = resultados_fin[idx_optimo]

# 6. PANEL VISUAL DE RESULTADOS
# ------------------------------------------------------------------------------
# A. Gráfico del Estadístico KS (Poder de Discriminación)
plt.figure(figsize=(10, 6))
plt.plot(thresholds, tpr, label='Tasa de Verdaderos Positivos (Morosos)', color='red', lw=2)
plt.plot(thresholds, fpr, label='Tasa de Falsos Positivos (Sanos)', color='blue', lw=2)
plt.axvline(thresholds[np.argmax(tpr - fpr)], color='black', linestyle='--', label=f'KS: {ks_stat:.3f}')
plt.xlim([0, 1])
plt.title('Estadístico KS: Capacidad de Separación del Modelo', fontsize=14)
plt.xlabel('Umbral de Probabilidad de Default')
plt.ylabel('Proporción Acumulada')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
print("\n")

# B. Curva de Rentabilidad (Optimización de Beneficio)
plt.figure(figsize=(10, 6))
plt.plot(umbrales_busqueda, beneficios, color='green', lw=3)
plt.axvline(x=umbral_optimo, color='red', linestyle='--', label=f'Corte Óptimo ({umbral_optimo*100:.1f}%)')
plt.title('Maximización de Rentabilidad (Profit Curve)', fontsize=14)
plt.xlabel('Umbral de Aprobación (Prob. Default)')
plt.ylabel('Beneficio Neto ($)')
plt.gca().yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, pos: f'${x*1e-6:.1f}M'))
plt.legend(loc='lower center')
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()
print("\n")

# C. Matriz de Confusión Financiera
plt.figure(figsize=(8, 6))
y_pred_opt = (y_probs >= umbral_optimo).astype(int)
cm = confusion_matrix(y_test, y_pred_opt)
ConfusionMatrixDisplay(cm, display_labels=['Aprobar (Sano)', 'Rechazar (Riesgo)']).plot(cmap='Blues', ax=plt.gca())
plt.title(f"Impacto Operativo (Umbral: {umbral_optimo*100:.1f}%)", fontsize=14)
plt.grid(False) # Las matrices de confusión no suelen llevar grid
plt.show()
print("\n")

# 7. EXPLICABILIDAD (SHAP)
# ------------------------------------------------------------------------------
print("\n⏳ 7. Generando Explicabilidad del Modelo (SHAP)...")
explainer = shap.TreeExplainer(model_xgb)
shap_values = explainer.shap_values(X_test_imp)

plt.figure(figsize=(10, 5))
shap.summary_plot(shap_values, X_test_imp, plot_type="dot", max_display=7, show=False)
plt.title("Factores Clave en la Decisión Crediticia (Top 7)", fontsize=14)
plt.show()

# 8. PRESENTACIÓN DE RESULTADOS EJECUTIVOS (TIPO DATA SCIENCE JR/MID)
# ------------------------------------------------------------------------------
print(f"\n{'='*70}")
print(f"{' 📈 REPORTE EJECUTIVO DE RIESGOS Y RENTABILIDAD ':^70}")
print(f"{'='*70}\n")

print("🔍 1. RESUMEN ESTADÍSTICO (Poder Predictivo)")
print(f"  • Coeficiente Gini: {gini:.4f} (Estándar de mercado cumplido)")
print(f"  • Estadístico KS:   {ks_stat:.4f} @ Umbral {ks_threshold*100:.1f}%")
print("    Explicación: El modelo separa excelentemente a los buenos de los malos pagadores.\n")

print("🎯 2. INTERPRETACIÓN DE LA MATRIZ DE CONFUSIÓN (Al umbral de máxima ganancia)")
print(f"  Bajo la política de rechazar a clientes con Riesgo > {umbral_optimo*100:.1f}%:")
print(f"  ✅ Sanos Aprobados (TN):        {cm[0][0]:,} -> Clientes que generan ingresos por intereses.")
print(f"  ❌ Fraude Evitado (TP):         {cm[1][1]:,} -> Morosos bloqueados. Capital protegido.")
print(f"  ⚠️ Falsos Positivos (FP):       {cm[0][1]:,} -> Buenos clientes rechazados (Costo de oportunidad).")
print(f"  🩸 Falsos Negativos (FN):       {cm[1][0]:,} -> Morosos que se filtraron (Pérdida real de LGD).\n")

print("💰 3. RESULTADOS FINANCIEROS DETALLADOS (P&L)")
print("  Proyección sobre la cartera de validación:")
print(f"  [+] Intereses Ganados (Sanos):       + ${ganancia_int:,.2f}")
print(f"  [-] Capital Perdido (Default LGD):   - ${perdida_def:,.2f}")
print("-" * 50)
print(f"  🚀 BENEFICIO NETO MAXIMIZADO:        = ${net_profit:,.2f}")
print("-" * 50)
print(f"  🛡️ Capital Salvado (Fraude Evitado):   ${cap_salvado:,.2f}")
print(f"  📉 Dinero dejado en la mesa (FP):      ${costo_op:,.2f}")
print(f"\n{'='*70}")